# 02 · Limpieza del Dataset
## Proyecto StyleNow Analytics
En este notebook limpiamos el dataset sucio y lo preparamos para el análisis.

In [1]:
# Importamos librerías
import pandas as pd
import numpy as np

# Leemos el dataset sucio
df = pd.read_csv('../data/raw/stylenow_raw.csv')

print(f"Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas")

Dataset cargado: 100200 filas y 9 columnas


#### Exploración inicial del dataset

In [2]:
# Información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100200 entries, 0 to 100199
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id_pedido   100200 non-null  object 
 1   id_cliente  100200 non-null  object 
 2   fecha       100200 non-null  object 
 3   categoria   99200 non-null   object 
 4   producto    100200 non-null  object 
 5   precio      99199 non-null   float64
 6   descuento   99199 non-null   float64
 7   cantidad    100200 non-null  int64  
 8   pais        99197 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 6.9+ MB


In [3]:
df.sample(20)

,id_pedido,id_cliente,fecha,categoria,producto,precio,descuento,cantidad,pais
35576,PED45576,CLI1226,2023-07-17 00:00:00,Accesorios,Bufanda Lana,38.78,0.0,3,Francia
65574,PED75574,CLI1420,2024-12-05 00:00:00,Zapatos,Tacón Nude,87.88,0.0,2,Francia
89424,PED99424,CLI1153,2023-08-12 00:00:00,Accesorios,Bufanda Lana,33.47,15.0,1,Portugal
8761,PED18761,CLI1484,2025-02-07 00:00:00,Zapatos,Sandalia Plana,79.64,10.0,3,España
41353,PED51353,CLI1411,2024-06-10 00:00:00,Vestidos,Vestido Verano,79.22,0.0,3,España
67750,PED77750,CLI1111,2024-03-13 00:00:00,Vestidos,Vestido Floral,85.19,5.0,4,Alemania
19726,PED29726,CLI1340,sin fecha,Accesorios,Bufanda Lana,41.94,25.0,2,España
35619,PED45619,CLI1146,2023-10-12 00:00:00,Zapatos,Zapatilla Blanca,95.96,10.0,2,Alemania
23945,PED33945,CLI1063,2024-03-19 00:00:00,Accesorios,Bufanda Lana,21.81,20.0,1,Portugal
70306,PED80306,CLI1024,2025-02-09 00:00:00,Accesorios,Bolso Mini,41.68,20.0,2,Alemania


In [4]:
# Resumen estadístico de las columnas numéricas
df.describe()

,precio,descuento,cantidad
count,99199.000000,99199.000000,100200.000000
mean,55.940618,8.338088,2.498463
std,25.454274,9.280350,1.118966
min,-108.200000,0.000000,1.000000
25%,33.060000,0.000000,1.000000
50%,54.930000,5.000000,3.000000
75%,78.550000,15.000000,3.000000
max,121.540000,150.000000,4.000000


In [4]:
# Resumen de nulos por columna
print("Nulos por columna:")
print(df.isnull().sum())
print(f"\nTotal de nulos: {df.isnull().sum().sum()}")

# Duplicados
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

Nulos por columna:
id_pedido        0
id_cliente       0
fecha            0
categoria     1000
producto         0
precio        1001
descuento     1001
cantidad         0
pais          1003
dtype: int64

Total de nulos: 4005

Filas duplicadas: 150


#### Orden de limpieza
1. Eliminar duplicados
2. Corregir tipos de datos
3. Limpiar texto
4. Corregir outliers imposibles
5. Tratar nulos

In [ ]:
#### Limpieza 1: Eliminación de duplicados

In [6]:
# Eliminamos filas duplicadas
df_clean = df.drop_duplicates()

print(f"Filas antes: {df.shape[0]}")
print(f"Filas después de eliminar duplicados: {df_clean.shape[0]}")
print(f"Duplicados eliminados: {df.shape[0] - df_clean.shape[0]}")


Filas antes: 100200
Filas después de eliminar duplicados: 100050
Duplicados eliminados: 150


In [ ]:
# ID Pedido al ser primary key no deberia tener filas duplicadas
df_clean.duplicated(subset=['id_pedido']).sum()

np.int64(50)

In [11]:
# Eliminamos duplicados por id_pedido, quedándonos con la primera ocurrencia
df_clean = df_clean.drop_duplicates(subset=['id_pedido'],keep='first')
print(f"Filas después de eliminar duplicados por id_pedido: {df_clean.shape[0]}")


Filas después de eliminar duplicados por id_pedido: 100000


#### Limpieza 2: Corrección de tipos d edatos

In [12]:
print("Tipos actuales")
print(df_clean.dtypes)

Tipos actuales
id_pedido      object
id_cliente     object
fecha          object
categoria      object
producto       object
precio        float64
descuento     float64
cantidad        int64
pais           object
dtype: object


In [13]:
# Vemos cuantas fechas tienen el valor 'sin fecha'
print("Fechas sin fecha:")
print(df_clean[df_clean["fecha"] == "sin fecha"].shape[0])

# Convertimos 'sin fecha' a nan para poder rconvertir la columna a datetime
df_clean["fecha"] = df_clean["fecha"].replace("sin fecha",np.nan)

# Convertirmos a datetime
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], errors='coerce')
print("\nTipo de fecha ahora:", df_clean['fecha'].dtype)
print("Nulos en fecha:",df_clean['fecha'].isnull().sum())

Fechas sin fecha:
7777

Tipo de fecha ahora: datetime64[ns]
Nulos en fecha: 7777


#### Limpieza 3: Normalización de texto en categoría

In [ ]:
print("Valores en categoria antes de limpiar:")
print(df_clean["categoria"].value_counts())

Valores unicos en categoria antes de limpiar:
categoria
Pantalones     18975
Camisetas      18941
Vestidos       18764
Zapatos        18736
Accesorios     18689
ACCESORIOS       628
VESTIDOS         586
ZAPATOS          571
PANTALONES       570
CAMISETAS        563
 Pantalones      424
 Camisetas       387
 Vestidos        378
 Accesorios      370
 Zapatos         368
 ZAPATOS          14
 VESTIDOS         11
 ACCESORIOS       10
 CAMISETAS         8
 PANTALONES        7
Name: count, dtype: int64


In [17]:
# Quitamos espacios en blanco y convertimos a minúsculas
df_clean["categoria"] = df_clean["categoria"].str.strip().str.title()
print("\nValores en categoria después de limpiar:")
print(df_clean["categoria"].value_counts())


Valores en categoria después de limpiar:
categoria
Pantalones    19976
Camisetas     19899
Vestidos      19739
Accesorios    19697
Zapatos       19689
Name: count, dtype: int64


#### Limpieza 4: Tratamientos de nulos

In [20]:
# Resumen de nulos actuales
print("Nulos por columna:")
print(df_clean.isnull().sum())

Nulos por columna:
id_pedido        0
id_cliente       0
fecha         7777
categoria     1000
producto         0
precio        1000
descuento     1000
cantidad         0
pais          1000
dtype: int64


In [ ]:
# Verificamos si los nulos coinciden en las mismas filas
nulos_categoria = set(df_clean[df_clean["categoria"].isnull()].index)
nulos_precio = set(df_clean[df_clean["precio"].isnull()].index)
nulos_descuento =  set(df_clean[df_clean["descuento"].isnull()].index)
nulos_pais = set(df_clean[df_clean["pais"].isnull()].index)
nulos_fecha = set(df_clean[df_clean['fecha'].isnull()].index)

print(f"Nulos solo en fecha: {len(nulos_fecha - nulos_categoria - nulos_precio - nulos_descuento - nulos_pais)}")
print(f"Nulos solo en precio: {len(nulos_precio - nulos_categoria - nulos_fecha - nulos_descuento - nulos_pais)}")
print(f"Nulos solo en precio: {len(nulos_descuento - nulos_categoria - nulos_fecha - nulos_precio - nulos_pais)}")
print(f"Nulos solo en categoria: {len(nulos_categoria - nulos_precio - nulos_descuento - nulos_fecha - nulos_pais)}")
print(f"Nulos en todas las columnas a la vez: {len(nulos_categoria & nulos_precio & nulos_descuento & nulos_fecha & nulos_pais)}")
print(f"Nulos compartidos categoria y precio: {len(nulos_categoria & nulos_precio)}")

Nulos solo en fecha: 7475
Nulos solo en precio: 896
Nulos solo en precio: 894
Nulos solo en categoria: 898
Nulos en todas las columnas a la vez: 0
Nulos compartidos categoria y precio: 11
Nulos compartidos categoria y precio: 8


Tratamiento de nulos - Criterios por columna:
- **fecha**: eliminar filas → sin fecha no hay análisis temporal
- **precio**: imputar con la mediana del grupo producto → más robusto que la media
- **categoria**: imputar mediante el producto → cada producto pertenece a una única categoría
- **descuento**: rellenar con 0 → si no hay descuento registrado asumimos que no hubo
- **pais**: rellenar con 'Desconocido' → no perdemos el pedido pero no inventamos el dato

In [31]:
df_clean = df_clean.dropna(subset=["fecha"])
print(f"Filas después de eliminar nulos en fecha: {df_clean.shape[0]}")

Filas después de eliminar nulos en fecha: 92223


In [32]:
# Rellenamos nulos en descuento con 0 (asumiendo que no hay descuento)
df_clean["descuento"] = df_clean["descuento"].fillna(0)
print(f"Nulos en descuento: {df_clean['descuento'].isnull().sum()}")

Nulos en descuento: 0


In [33]:
# Rellenamos pais nulos con "Desconocido"
df_clean["pais"] = df_clean["pais"].fillna("Desconocido")
print(f"Nulos en pais: {df_clean["pais"].isnull().sum()}")
print(df_clean['pais'].value_counts())

Nulos en pais: 0
pais
Portugal       18317
Francia        18304
Italia         18270
España         18241
Alemania       18162
Desconocido      929
Name: count, dtype: int64


In [35]:
# Creamos un diccionario de mapeo producto -> categoria
mapeo_categoria = {
    'Camiseta Básica': 'Camisetas', 'Polo Slim': 'Camisetas', 
    'Camiseta Estampada': 'Camisetas', 'Camiseta Oversize': 'Camisetas', 
    'Top Tirantes': 'Camisetas',
    'Vaquero Slim': 'Pantalones', 'Chino Beige': 'Pantalones', 
    'Pantalón Casual': 'Pantalones', 'Jogger Negro': 'Pantalones', 
    'Vaquero Wide Leg': 'Pantalones',
    'Vestido Floral': 'Vestidos', 'Vestido Negro': 'Vestidos', 
    'Vestido Verano': 'Vestidos', 'Vestido Midi': 'Vestidos', 
    'Vestido Lencero': 'Vestidos',
    'Zapatilla Blanca': 'Zapatos', 'Bota Marrón': 'Zapatos', 
    'Sandalia Plana': 'Zapatos', 'Tacón Nude': 'Zapatos', 
    'Deportiva Runner': 'Zapatos',
    'Bolso Mini': 'Accesorios', 'Cinturón Cuero': 'Accesorios', 
    'Gafas Sol': 'Accesorios', 'Sombrero Paja': 'Accesorios', 
    'Bufanda Lana': 'Accesorios'
}

# Rellenamos la categoria nula usando el producto
df_clean["categoria"] = df_clean.apply(
    lambda row: mapeo_categoria[row["producto"]] if pd.isnull(row["categoria"]) else row["categoria"],
    axis=1
)

print(f"Nulos en categoria: {df_clean['categoria'].isnull().sum()}")

Nulos en categoria: 0


In [37]:
# Imputamos precio nulo con la mediana del mismo producto
df_clean["precio"] = df_clean.groupby("producto")["precio"].transform( lambda x: x.fillna(x.median()))

print(f"Nulos en precio: {df_clean['precio'].isnull().sum()}")

Nulos en precio: 0
